In [1]:
import sys
import os
notebook_dir = os.path.dirname(os.path.abspath(''))
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..', 'lime_ndt')))
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..')))

## California Housing Dataset

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# LIME
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
from lime_ndt.lime_tabular import LimeTabularExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0
        return self

# ========================
# Charger dataset complet
# ========================
data = fetch_california_housing()
X, y = data.data, data.target
feature_names = data.feature_names
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# Modèle global (MLP)
# ========================
mlp_global = MLPRegressor(hidden_layer_sizes=(128, 64, 32),
                          activation='relu',
                          solver='adam',
                          max_iter=2000,
                          random_state=42)
mlp_global.fit(X_train, y_train)
predict_fn = mlp_global.predict

# ========================
# Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train, feature_names=feature_names,
    discretize_continuous=False, mode='regression'
)
explainer_ndt = LimeNDTExplainer(
    X_train, feature_names=feature_names,
    discretize_continuous=False, mode='regression'
)

# ========================
# Fonction d'explication
# ========================
def get_explanation_vector(explainer, instance, predict_fn, local_model):
    exp = explainer.explain_instance(instance, predict_fn, model_regressor=local_model)
    weights = dict(exp.local_exp[1])
    return np.array([weights.get(i, 0.0) for i in range(len(feature_names))])

# ========================
# Fonction de régularité
# ========================
def compute_regularity(explainer, local_model_cls, X_data, k=5, save_path=None):
    # Ensure X_data is a numpy array (iterating a pandas DataFrame yields column names)
    if hasattr(X_data, 'values'):
        X_np = X_data.values
    else:
        X_np = np.asarray(X_data)

    n = len(X_np)
    E = np.zeros((n, X_np.shape[1]))

    print(f"→ Génération des {n} explications locales...")
    for i, x in enumerate(tqdm(X_np)):
        try:
            E[i] = get_explanation_vector(explainer, x, predict_fn, local_model_cls())
        except Exception as e:
            # si LIME échoue sur une instance, mettre un vecteur nul
            print(f"⚠️ Instance {i} skipped ({e})")
            E[i] = np.zeros(X_np.shape[1])

    # Remplacer NaN et inf par 0 pour éviter les erreurs de similarité
    E = np.nan_to_num(E, nan=0.0, posinf=0.0, neginf=0.0)

    if save_path:
        np.save(save_path, E)
        print(f"Explanations saved to {save_path}")

    print("→ Calcul des similarités cosinus entre voisins...")
    nbrs = NearestNeighbors(n_neighbors=k+1).fit(X_np)
    _, indices = nbrs.kneighbors(X_np)
    indices = indices[:, 1:]

    cos_sims = []
    for i, neigh_idx in enumerate(indices):
        # Extra safety: handle any all-zero vectors
        Ei = E[i].reshape(1, -1)
        En = E[neigh_idx]
        if np.all(Ei == 0) or np.all(En == 0):
            continue
        sims = cosine_similarity(Ei, En)[0]
        cos_sims.append(np.mean(sims))

    return np.mean(cos_sims) if len(cos_sims) > 0 else 0.0


# ========================
# Exécution sur tout X_test
# ========================
results = {}
results["LinearRegression"] = compute_regularity(explainer_classic, LinearRegression, X_test)
results["DecisionTree"] = compute_regularity(explainer_ndt, DecisionTreeWrapper, X_test)
results["NDT"] = compute_regularity(
    explainer_ndt,
    lambda: NDTRegressorWrapper(D=X_train.shape[1], gammas=[100, 1]),
    X_test
)

print("\n=== Régularité moyenne sur tout le jeu de test ===")
for model_name, score in results.items():
    print(f"{model_name}: {score:.3f}")


## Diabetes Dataset

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# LIME
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
from lime_ndt.lime_tabular import LimeTabularExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0
        return self

# ========================
# Charger dataset complet
# ========================
data = load_diabetes()
X, y = data.data, data.target
feature_names = data.feature_names
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# Modèle global (MLP)
# ========================
mlp_global = MLPRegressor(hidden_layer_sizes=(128, 64, 32),
                          activation='relu',
                          solver='adam',
                          max_iter=2000,
                          random_state=42)
mlp_global.fit(X_train, y_train)
predict_fn = mlp_global.predict

# ========================
# Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train, feature_names=feature_names,
    discretize_continuous=False, mode='regression'
)
explainer_ndt = LimeNDTExplainer(
    X_train, feature_names=feature_names,
    discretize_continuous=False, mode='regression'
)

# ========================
# Fonction d'explication
# ========================
def get_explanation_vector(explainer, instance, predict_fn, local_model):
    exp = explainer.explain_instance(instance, predict_fn, model_regressor=local_model)
    weights = dict(exp.local_exp[1])
    return np.array([weights.get(i, 0.0) for i in range(len(feature_names))])

# ========================
# Fonction de régularité
# ========================
def compute_regularity(explainer, local_model_cls, X_data, k=10, save_path=None):
    # Ensure X_data is a numpy array (iterating a pandas DataFrame yields column names)
    if hasattr(X_data, 'values'):
        X_np = X_data.values
    else:
        X_np = np.asarray(X_data)

    n = len(X_np)
    E = np.zeros((n, X_np.shape[1]))

    print(f"→ Génération des {n} explications locales...")
    for i, x in enumerate(tqdm(X_np)):
        try:
            E[i] = get_explanation_vector(explainer, x, predict_fn, local_model_cls())
        except Exception as e:
            # si LIME échoue sur une instance, mettre un vecteur nul
            print(f"⚠️ Instance {i} skipped ({e})")
            E[i] = np.zeros(X_np.shape[1])

    # Remplacer NaN et inf par 0 pour éviter les erreurs de similarité
    E = np.nan_to_num(E, nan=0.0, posinf=0.0, neginf=0.0)

    if save_path:
        np.save(save_path, E)
        print(f"Explanations saved to {save_path}")

    print("→ Calcul des similarités cosinus entre voisins...")
    nbrs = NearestNeighbors(n_neighbors=k+1).fit(X_np)
    _, indices = nbrs.kneighbors(X_np)
    indices = indices[:, 1:]

    cos_sims = []
    for i, neigh_idx in enumerate(indices):
        # Extra safety: handle any all-zero vectors
        Ei = E[i].reshape(1, -1)
        En = E[neigh_idx]
        if np.all(Ei == 0) or np.all(En == 0):
            continue
        sims = cosine_similarity(Ei, En)[0]
        cos_sims.append(np.mean(sims))

    return np.mean(cos_sims) if len(cos_sims) > 0 else 0.0


# ========================
# Exécution sur tout X_test
# ========================
results = {}
results["LinearRegression"] = compute_regularity(explainer_classic, LinearRegression, X_test)
results["DecisionTree"] = compute_regularity(explainer_ndt, DecisionTreeWrapper, X_test)
results["NDT"] = compute_regularity(
    explainer_ndt,
    lambda: NDTRegressorWrapper(D=X_train.shape[1], gammas=[100, 1]),
    X_test
)

print("\n=== Régularité moyenne sur tout le jeu de test ===")
for model_name, score in results.items():
    print(f"{model_name}: {score:.3f}")


## Ames Housing Dataset

In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# LIME
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
from lime_ndt.lime_tabular import LimeTabularExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0
        return self

# ========================
# Charger dataset complet
# ========================
data = fetch_openml(name='house_prices', as_frame=True)
X = data.data.select_dtypes(include=[np.number]).dropna(axis=1)
y = data.target.astype(float)
feature_names = data.feature_names
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# Modèle global (MLP)
# ========================
mlp_global = MLPRegressor(hidden_layer_sizes=(128, 64, 32),
                          activation='relu',
                          solver='adam',
                          max_iter=2000,
                          random_state=42)
mlp_global.fit(X_train, y_train)
predict_fn = mlp_global.predict

# ========================
# Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train, feature_names=feature_names,
    discretize_continuous=False, mode='regression'
)
explainer_ndt = LimeNDTExplainer(
    X_train, feature_names=feature_names,
    discretize_continuous=False, mode='regression'
)

# ========================
# Fonction d'explication
# ========================
def get_explanation_vector(explainer, instance, predict_fn, local_model):
    exp = explainer.explain_instance(instance, predict_fn, model_regressor=local_model)
    weights = dict(exp.local_exp[1])
    return np.array([weights.get(i, 0.0) for i in range(len(feature_names))])

# ========================
# Fonction de régularité
# ========================
def compute_regularity(explainer, local_model_cls, X_data, k=10, save_path=None):
    # Ensure X_data is a numpy array (iterating a pandas DataFrame yields column names)
    if hasattr(X_data, 'values'):
        X_np = X_data.values
    else:
        X_np = np.asarray(X_data)

    n = len(X_np)
    E = np.zeros((n, X_np.shape[1]))

    print(f"→ Génération des {n} explications locales...")
    for i, x in enumerate(tqdm(X_np)):
        try:
            E[i] = get_explanation_vector(explainer, x, predict_fn, local_model_cls())
        except Exception as e:
            # si LIME échoue sur une instance, mettre un vecteur nul
            print(f"⚠️ Instance {i} skipped ({e})")
            E[i] = np.zeros(X_np.shape[1])

    # Remplacer NaN et inf par 0 pour éviter les erreurs de similarité
    E = np.nan_to_num(E, nan=0.0, posinf=0.0, neginf=0.0)

    if save_path:
        np.save(save_path, E)
        print(f"Explanations saved to {save_path}")

    print("→ Calcul des similarités cosinus entre voisins...")
    nbrs = NearestNeighbors(n_neighbors=k+1).fit(X_np)
    _, indices = nbrs.kneighbors(X_np)
    indices = indices[:, 1:]

    cos_sims = []
    for i, neigh_idx in enumerate(indices):
        # Extra safety: handle any all-zero vectors
        Ei = E[i].reshape(1, -1)
        En = E[neigh_idx]
        if np.all(Ei == 0) or np.all(En == 0):
            continue
        sims = cosine_similarity(Ei, En)[0]
        cos_sims.append(np.mean(sims))

    return np.mean(cos_sims) if len(cos_sims) > 0 else 0.0


# ========================
# Exécution sur tout X_test
# ========================
results = {}
results["LinearRegression"] = compute_regularity(explainer_classic, LinearRegression, X_test)
results["DecisionTree"] = compute_regularity(explainer_ndt, DecisionTreeWrapper, X_test)
results["NDT"] = compute_regularity(
    explainer_ndt,
    lambda: NDTRegressorWrapper(D=X_train.shape[1], gammas=[100, 1]),
    X_test
)

print("\n=== Régularité moyenne sur tout le jeu de test ===")
for model_name, score in results.items():
    print(f"{model_name}: {score:.3f}")


## Iris Dataset

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# LIME
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
from lime_ndt.lime_tabular import LimeTabularExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# Wrapper pour DecisionTree (classifier)
# ========================
class DecisionTreeRegressorWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        # Simuler des attributs que LIME attend
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0.0
        return self

# ========================
# Charger dataset complet
# ========================
data = load_iris()
X, y = data.data, data.target
feature_names = data.feature_names
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# Modèle global (MLP classifier)
# ========================
mlp_global = MLPClassifier(hidden_layer_sizes=(128, 64, 32),
                          activation='relu',
                          solver='adam',
                          max_iter=2000,
                          random_state=42)
mlp_global.fit(X_train, y_train)
# LIME expects a predict function that returns probabilities for classification
predict_fn = mlp_global.predict_proba

# ========================
# Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train, feature_names=feature_names,
    discretize_continuous=False, mode='classification'
 )
explainer_ndt = LimeNDTExplainer(
    X_train, feature_names=feature_names,
    discretize_continuous=False, mode='classification'
 )

# ========================
# Fonction d'explication
# ========================
def get_explanation_vector(explainer, instance, predict_fn, local_model):
    """Return the explanation weight vector for the predicted class.
    For classification LIME returns a dict mapping class -> list[(feature_idx, weight)].
    We choose the class predicted by the global model (predict_fn) for the instance.
    """
    exp = explainer.explain_instance(instance, predict_fn, model_regressor=local_model)
    # predict_fn is expected to return class probabilities for classification
    try:
        probs = predict_fn(instance.reshape(1, -1))
        if probs.ndim == 2:
            pred_class = int(np.argmax(probs, axis=1)[0])
        else:
            # fallback: if predict_fn returns labels
            pred_class = int(probs)
    except Exception:
        # if predict_fn fails, default to class 0
        pred_class = 0
    # exp.local_exp is a dict keyed by class
    weights = dict(exp.local_exp.get(pred_class, []))
    return np.array([weights.get(i, 0.0) for i in range(len(feature_names))])

# ========================
# Fonction de régularité
# ========================
def compute_regularity(explainer, local_model_cls, X_data, k=1, save_path=None):
    # Ensure X_data is a numpy array (iterating a pandas DataFrame yields column names)
    if hasattr(X_data, 'values'):
        X_np = X_data.values
    else:
        X_np = np.asarray(X_data)

    n = len(X_np)
    E = np.zeros((n, X_np.shape[1]))

    print(f"→ Génération des {n} explications locales...")
    for i, x in enumerate(tqdm(X_np)):
        try:
            E[i] = get_explanation_vector(explainer, x, predict_fn, local_model_cls())
        except Exception as e:
            # si LIME échoue sur une instance, mettre un vecteur nul
            print(f"⚠️ Instance {i} skipped ({e})")
            E[i] = np.zeros(X_np.shape[1])

    # Remplacer NaN et inf par 0 pour éviter les erreurs de similarité
    E = np.nan_to_num(E, nan=0.0, posinf=0.0, neginf=0.0)

    if save_path:
        np.save(save_path, E)
        print(f"Explanations saved to {save_path}")

    print("→ Calcul des similarités cosinus entre voisins...")
    nbrs = NearestNeighbors(n_neighbors=k+1).fit(X_np)
    _, indices = nbrs.kneighbors(X_np)
    indices = indices[:, 1:]

    cos_sims = []
    for i, neigh_idx in enumerate(indices):
        # Extra safety: handle any all-zero vectors
        Ei = E[i].reshape(1, -1)
        En = E[neigh_idx]
        if np.all(Ei == 0) or np.all(En == 0):
            continue
        sims = cosine_similarity(Ei, En)[0]
        cos_sims.append(np.mean(sims))

    return np.mean(cos_sims) if len(cos_sims) > 0 else 0.0

# ========================
# Exécution sur tout X_test
# ========================
results = {}
results["LinearRegression"] = compute_regularity(explainer_classic, LinearRegression, X_test)
results["DecisionTreeRegressor"] = compute_regularity(explainer_classic, DecisionTreeRegressorWrapper, X_test)
results["NDT"] = compute_regularity(explainer_classic, lambda: NDTRegressorWrapper(D=X_train.shape[1], gammas=[100, 1]), X_test)

print("\n=== Régularité moyenne sur tout le jeu de test ===")
for model_name, score in results.items():
    print(f"{model_name}: {score:.3f}")

## Wine Dataset

In [9]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# LIME
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
from lime_ndt.lime_tabular import LimeTabularExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# Wrapper pour DecisionTree (classifier)
# ========================
class DecisionTreeRegressorWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        # Simuler des attributs que LIME attend
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0.0
        return self

# ========================
# Charger dataset complet
# ========================
data = load_wine()
X, y = data.data, data.target
feature_names = data.feature_names
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# Modèle global (MLP classifier)
# ========================
mlp_global = MLPClassifier(hidden_layer_sizes=(128, 64, 32),
                          activation='relu',
                          solver='adam',
                          max_iter=2000,
                          random_state=42)
mlp_global.fit(X_train, y_train)
# LIME expects a predict function that returns probabilities for classification
predict_fn = mlp_global.predict_proba

# ========================
# Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train, feature_names=feature_names,
    discretize_continuous=False, mode='classification'
 )
explainer_ndt = LimeNDTExplainer(
    X_train, feature_names=feature_names,
    discretize_continuous=False, mode='classification'
 )

# ========================
# Fonction d'explication
# ========================
def get_explanation_vector(explainer, instance, predict_fn, local_model):
    """Return the explanation weight vector for the predicted class.
    For classification LIME returns a dict mapping class -> list[(feature_idx, weight)].
    We choose the class predicted by the global model (predict_fn) for the instance.
    """
    exp = explainer.explain_instance(instance, predict_fn, model_regressor=local_model)
    # predict_fn is expected to return class probabilities for classification
    try:
        probs = predict_fn(instance.reshape(1, -1))
        if probs.ndim == 2:
            pred_class = int(np.argmax(probs, axis=1)[0])
        else:
            # fallback: if predict_fn returns labels
            pred_class = int(probs)
    except Exception:
        # if predict_fn fails, default to class 0
        pred_class = 0
    # exp.local_exp is a dict keyed by class
    weights = dict(exp.local_exp.get(pred_class, []))
    return np.array([weights.get(i, 0.0) for i in range(len(feature_names))])

# ========================
# Fonction de régularité
# ========================
def compute_regularity(explainer, local_model_cls, X_data, k=1, save_path=None):
    # Ensure X_data is a numpy array (iterating a pandas DataFrame yields column names)
    if hasattr(X_data, 'values'):
        X_np = X_data.values
    else:
        X_np = np.asarray(X_data)

    n = len(X_np)
    E = np.zeros((n, X_np.shape[1]))

    print(f"→ Génération des {n} explications locales...")
    for i, x in enumerate(tqdm(X_np)):
        try:
            E[i] = get_explanation_vector(explainer, x, predict_fn, local_model_cls())
        except Exception as e:
            # si LIME échoue sur une instance, mettre un vecteur nul
            print(f"⚠️ Instance {i} skipped ({e})")
            E[i] = np.zeros(X_np.shape[1])

    # Remplacer NaN et inf par 0 pour éviter les erreurs de similarité
    E = np.nan_to_num(E, nan=0.0, posinf=0.0, neginf=0.0)

    if save_path:
        np.save(save_path, E)
        print(f"Explanations saved to {save_path}")

    print("→ Calcul des similarités cosinus entre voisins...")
    nbrs = NearestNeighbors(n_neighbors=k+1).fit(X_np)
    _, indices = nbrs.kneighbors(X_np)
    indices = indices[:, 1:]

    cos_sims = []
    for i, neigh_idx in enumerate(indices):
        # Extra safety: handle any all-zero vectors
        Ei = E[i].reshape(1, -1)
        En = E[neigh_idx]
        if np.all(Ei == 0) or np.all(En == 0):
            continue
        sims = cosine_similarity(Ei, En)[0]
        cos_sims.append(np.mean(sims))

    return np.mean(cos_sims) if len(cos_sims) > 0 else 0.0

# ========================
# Exécution sur tout X_test
# ========================
results = {}
results["LinearRegression"] = compute_regularity(explainer_classic, LinearRegression, X_test)
results["DecisionTreeRegressor"] = compute_regularity(explainer_classic, DecisionTreeRegressorWrapper, X_test)
results["NDT"] = compute_regularity(explainer_classic, lambda: NDTRegressorWrapper(D=X_train.shape[1], gammas=[100, 1]), X_test)

print("\n=== Régularité moyenne sur tout le jeu de test ===")
for model_name, score in results.items():
    print(f"{model_name}: {score:.3f}")

→ Génération des 45 explications locales...


100%|██████████| 45/45 [00:01<00:00, 39.49it/s]


→ Calcul des similarités cosinus entre voisins...
→ Génération des 45 explications locales...


100%|██████████| 45/45 [00:06<00:00,  6.75it/s]


→ Calcul des similarités cosinus entre voisins...
→ Génération des 45 explications locales...


...

⚠️ Instance 40 skipped (Input 0 of layer "functional_918" is incompatible with the layer: expected shape=(None, 13), found shape=(None, 10))
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)
⚠️ Instance 41 skipped (Input 0 of layer "functional_921" is incompatible with the layer: expected shape=(None, 13), found shape=(None, 10))


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(
 96%|█████████▌| 43/45 [00:09<00:00,  4.91it/s]

mean_leaf_values shape: (31, 1, 1)
self.L: 31 self.C: 1
mean_leaf_values shape after squeeze: (31, 1)
⚠️ Instance 42 skipped (Input 0 of layer "functional_924" is incompatible with the layer: expected shape=(None, 13), found shape=(None, 10))
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(
 98%|█████████▊| 44/45 [00:09<00:00,  4.94it/s]c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(
100%|██████████| 45/45 [00:09<00:00,  4.63it/s]

⚠️ Instance 43 skipped (Input 0 of layer "functional_927" is incompatible with the layer: expected shape=(None, 13), found shape=(None, 10))
mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)
⚠️ Instance 44 skipped (Input 0 of layer "functional_930" is incompatible with the layer: expected shape=(None, 13), found shape=(None, 10))
→ Calcul des similarités cosinus entre voisins...

=== Régularité moyenne sur tout le jeu de test ===
LinearRegression: 0.987
DecisionTreeRegressor: 0.998
NDT: 0.000
